# Start vespa

```bash
docker run -d \
  --name vespa \
  -p 8080:8080 \
  -p 19071:19071 \
  -v /home/user/docker/volumes/vespa:/vespa \
  vespaengine/vespa
````



# Test

```bash
curl http://localhost:19071/state/v1/health
```

## Application

Minimal application:

```
app/
├── services.xml
└── schemas/
    └── doc.sd
```


### Services.xml

```xml
<services version="1.0">
  <container id="default" version="1.0">
    <search/>
    <document-api/>
  </container>

  <content id="content" version="1.0">
    <redundancy>1</redundancy>

    <documents>
      <document type="doc" mode="index"/>
    </documents>

    <nodes>
      <node distribution-key="0" hostalias="node1"/>
    </nodes>
  </content>
</services>
``` 

### doc.sd

```sd
schema doc {

  document doc {

    field id type string {
      indexing: summary | attribute
      index: enable-bm25
    }

    field title type string {
      indexing: index | summary
      index: enable-bm25
    }

    field body type string {
      indexing: index | summary
      index: enable-bm25
    }

    field embedding type tensor<float>(d[1024]) {
      indexing: attribute
    }
  }
}
```

Deploy:

```bash
user@fedora:~/src/playbook/vespa/simple_app$ vespa deploy . -a local.simple_app.0
```

# Insert a document

```bash
curl -H "Content-Type: application/json" \
     -X POST \
     --data '
{
  "fields": {
    "id": "1",
    "title": "Introduction to Vespa",
    "body": "Vespa is a scalable search engine"
  }
}' \
http://localhost:8080/document/v1/doc/doc/docid/1
``` 


# Retrieval

Id:
```bash
curl \
http://localhost:8080/document/v1/doc/doc/docid/1
```

Full search test:

```bash
curl \
"http://localhost:8080/search/?yql=select%20*%20from%20sources%20*%20where%20userQuery()&query=search"
```

# Embedd document

In [ ]:
from datetime import datetime

from FlagEmbedding import BGEM3FlagModel
from numpy import size
# if running offline: export HF_HUB_OFFLINE=1
model = BGEM3FlagModel('BAAI/bge-m3')                        


sentences_1 = ["What is BGE M3?", "Defination of BM25"]
sentences_2 = ["BGE M3 is an embedding model supporting dense retrieval, lexical matching and multi-vector interaction.",
               "BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document"]

output = model.encode(sentences_1, return_dense=True, return_sparse=True, return_colbert_vecs=True)
now = datetime.now()
dense, sparse, multiv = output['dense_vecs'], output['lexical_weights'], output['colbert_vecs']
completed = datetime.now()

print(f"Completed in {completed-now}")

print("---------DENSE--------")
print(size(dense[0]))
print("---------SPARSE-------")
print(size(sparse[0]))
print("---------MULTI---------")
for i, v in enumerate(multiv):
    print(f"{i}: {size(v)}")
    


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Completed in 0:00:00.000111
---------DENSE--------
1024
---------SPARSE-------
1
---------MULTI---------
0: 8192
1: 7168


In [16]:
from vespa.application import Vespa

app = Vespa(url="http://localhost", port=8080)

doc = {
    "fields": {
        "id": "doc-1",
        "title": "My first document",
        "body": "This is the document body.",
        "embedding": dense[0].astype("float32").tolist()
    }
}

response = app.feed_data_point(
    data_id="doc-1",
    fields=doc["fields"],
    schema="doc"
)

print(response.status_code)
print(response.json)

200
{'id': 'id:doc:doc::doc-1', 'pathId': '/document/v1/doc/doc/docid/doc-1'}


#### Get document from ID

In [1]:
from vespa.application import Vespa

app = Vespa(url="http://localhost", port=8080)

response = app.get_data(
    schema="doc",
    data_id="doc-1"
)

print(response.json)

{'id': 'id:doc:doc::doc-1', 'pathId': '/document/v1/doc/doc/docid/doc-1'}


#### Select all documents

In [2]:
from vespa.application import Vespa

app = Vespa(url="http://localhost", port=8080)

response = app.query(
    yql="select id,title from doc where true",
    hits=100
)

for hit in response.hits:
    print(hit["fields"]["id"])
    print(hit["fields"]["title"])
    print(hit.get("relevance"))

doc-001
introduction to vespa search engine
0.0
doc-002
how to configure bm25 ranking in vespa
0.0
doc-003
dense vector retrieval with document embeddings
0.0
doc-004
hybrid search strategy combining bm25 and embeddings
0.0
doc-005
best practices for embedding normalization
0.0
doc-006
how to evaluate search relevance with mrr and recall
0.0
doc-007
understanding cosine similarity and angular distance
0.0
doc-008
question answering systems and passage retrieval
0.0
doc-009
using fieldsets for text and semantic data in vespa
0.0
doc-010
how to build a faq ranking dataset
0.0
doc-011
handling phrase queries and exact match search
0.0
doc-012
using title and body embeddings separately in vespa
0.0
doc-013
combining lexical and semantic search across product catalogs
0.0
doc-014
how embeddings improve semantic retrieval
0.0
doc-015
symptoms of high blood pressure and medical advice
0.0
doc-016
how to bake sourdough bread and maintain a starter
0.0


#### Get document count

In [3]:
from vespa.application import Vespa

app = Vespa(url="http://localhost", port=8080)

response = app.query(
    yql="select * from doc where true",
    hits=0 # We only want the total count of documents, not the actual documents
)

print(response.json["root"]["fields"]["totalCount"])

16


# Keyword search 

#### Search a specific field

In [4]:
from vespa.application import Vespa

app = Vespa(url="http://localhost", port=8080)

response = app.query(
    yql="select * from doc where title contains 'My first document'",
    hits=10,
)

for hit in response.hits:
    print(hit["id"])
    print(hit["fields"]["title"])
    print(hit.get("relevance"))

# Vector search

In [33]:
from FlagEmbedding import BGEM3FlagModel
import torch

model = BGEM3FlagModel('BAAI/bge-m3') 
app = Vespa(url="http://localhost", port=8080)




def get_embedding(text: str):
    dense = model.encode([text], return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
    return dense[0].astype("float32").tolist()

def get_query_embedding(text: str):
    dense = model.encode([text], return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
    tensor = torch.from_numpy(dense[0]).float()
    tensor = tensor.cpu()
    return tensor.numpy()
    
    #return dense[0].numpy() # fails ndarray
    #return dense[0].tolist() #query(q)' to be a tensor, but it is a string. Unless you set this to a string in code,
    #return torch.from_numpy(dense[0]).float() #Object of type Tensor is not JSON serializable
    # return dense[0] # ndarray is not serialzable



def feed_document(vespa, doc_id, title, body):
    text = title + " " + body
    embedding = get_embedding(text)

    doc = {
        "id": doc_id,
        "title": title,
        "body": body,
        "embedding": embedding
    }

    vespa.feed_data_point(
        schema="doc",
        data_id=doc_id,
        fields=doc
    )
    
# feed_document(
#     app,
#     doc_id="2",
#     title="Cat drinks milk",
#     body="The age old story about cat that refuses to drink milk if it is sour"
# )



In [36]:
query_text = "fox"

def get_embedding_new(text: str):
    dense = model.encode([text], return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
    vec = dense[0]
    vec = vec / torch.norm(torch.tensor(vec))
    return vec.tolist()

query_vector = get_embedding_new(query_text)

results = app.query(
    yql="""
        select *
        from doc
        where ({targetHits:10}nearestNeighbor(embedding, q))
    """,
    ranking="semantic",
    body={
        "input.query(q)": query_vector,
    },
)

# query = {
#     "yql": 'select id, title, body from doc where ([{"targetHits": 10}]nearestNeighbor(embedding, q))',
#     "hits": 10,
#     "input.query(q)": query_vector
# }

# results = app.query(query)
for hit in results.hits:
    print(hit["fields"]["title"])

Cat drinks milk
My first document


In [39]:
# Hybrid

results = app.query(
    yql="""
        select *
        from doc
        where ({targetHits:10}nearestNeighbor(embedding, q))
    """,
    ranking="hybrid",
    body={
        "input.query(q)": query_vector,
    },
)

for hit in results.hits:
    print(hit["fields"]["title"])

Cat drinks milk
My first document


---

# Vector options

```yaml
field body_embedding type tensor<float>(d[1024]) {
    indexing: attribute | index
    attribute {
        distance-metric: angular
    }
}
```

### Indexing
| Option      | Purpose                                                                                          |
| ----------- | ------------------------------------------------------------------------------------------------ |
| `attribute` | Stores the tensor in memory for fast lookup and ranking. Required for vector search.             |
| `index`     | Builds an ANN index (HNSW) over the vectors. Required for `nearestNeighbor()`.                   |
| `summary`   | Makes the tensor available in document summaries (usually avoided because embeddings are large). |

### Distance metric

| Metric                  | Meaning                                        | Typical embedding model                         |
| ----------------------- | ---------------------------------------------- | ----------------------------------------------- |
| `angular`               | Cosine similarity via angular distance         | Most sentence transformers                      |
| `euclidean`             | L2 distance                                    | Some image embeddings                           |
| `dotproduct`            | Dot product                                    | Models trained for Maximum Inner Product Search |
| `prenormalized-angular` | Cosine assuming vectors are already normalized | Normalized embeddings                           |


Typical vector field:

**indexing: attribute | index**

If you omit index, you can still compute closeness(field, body_embedding)

**during ranking**, but cannot use ANN search.

### HNSW Index

```yaml
field body_embedding type tensor<float>(d[1024]) {
    indexing: attribute | index

    attribute {
        distance-metric: angular
    }

    index {
        hnsw {
            max-links-per-node: 16
            neighbors-to-explore-at-insert: 200
        }
    }
}
```

| Option                           | Effect                     |
| -------------------------------- | -------------------------- |
| `max-links-per-node`             | Graph connectivity         |
| `neighbors-to-explore-at-insert` | Index construction quality |

Higher values improve recall but increase indexing cost.

---

# Query options

## Vespa Query Options for Vector and Hybrid Search

Vespa supports several query patterns depending on whether retrieval is lexical (BM25), semantic (vector search), or a combination of both.

### 1. Lexical Search (BM25)

Retrieves documents using an inverted index and BM25 scoring.

```sql
select * from doc where userQuery()
```

Typical rank profile:

```vespa
first-phase {
    expression: bm25(title) + bm25(body)
}
```

**Use case:** Traditional keyword search.

---

### 2. Pure Semantic Search (Nearest Neighbor)

Retrieves the nearest embedding vectors using the HNSW index.

```sql
select * from doc
where ({targetHits:100}nearestNeighbor(body_embedding, q))
```

Rank profile:

```vespa
first-phase {
    expression: closeness(field, body_embedding)
}
```

Requires:

```vespa
inputs {
    query(q) tensor<float>(d[1024])
}
```

and the query embedding:

```python
body["input.query(q)"] = query_vector
```

**Use case:** Semantic search where lexical overlap is not required.

---

### 3. Hybrid Search (BM25 + ANN Retrieval)

Retrieves candidates from both lexical and vector search.

```sql
select * from doc
where userQuery()
OR ({targetHits:100}nearestNeighbor(body_embedding, q))
```

Example rank profile:

```vespa
first-phase {
    expression:
        bm25(title)
      + bm25(body)
      + 0.8 * closeness(field, body_embedding)
}
```

**Use case:** Production retrieval systems combining keyword and semantic search.

---

### 4. BM25 Retrieval + Semantic Re-ranking

Retrieves documents lexically, then uses vector similarity during ranking.

```sql
select * from doc
where userQuery()
```

Rank profile:

```vespa
first-phase {
    expression:
        bm25(title)
      + bm25(body)
      + 0.8 * closeness(field, body_embedding)
}
```

This is the pattern used in the current implementation.

**Use case:** Improve BM25 ranking without performing ANN retrieval.

---

### 5. RANK Operator

Uses lexical retrieval while exposing vector features for ranking.

```sql
select * from doc
where userQuery()
RANK nearestNeighbor(body_embedding, q)
```

Ranking:

```vespa
first-phase {
    expression:
        bm25(body)
      + closeness(field, body_embedding)
}
```

**Use case:** Lexical retrieval with explicit nearest-neighbor ranking features.

---

### 6. Filtered Semantic Search

Combine metadata filters with ANN retrieval.

```sql
select * from doc
where category contains "science"
AND ({targetHits:100}nearestNeighbor(body_embedding, q))
```

**Use case:** Search within a subset of documents.

---

### 7. Multiple Embedding Fields

Search over multiple vector fields.

```sql
select * from doc
where
({targetHits:50}nearestNeighbor(title_embedding, q))
OR
({targetHits:50}nearestNeighbor(body_embedding, q))
```

Ranking can combine multiple similarity scores.

**Use case:** Documents with title, abstract, and body embeddings.

---

### 8. Multiple Query Embeddings

Use different query vectors for different fields.

```sql
select * from doc
where
nearestNeighbor(title_embedding, title_q)
OR
nearestNeighbor(body_embedding, body_q)
```

Rank profile:

```vespa
inputs {
    query(title_q) tensor<float>(d[1024])
    query(body_q) tensor<float>(d[1024])
}
```

**Use case:** Multi-modal or field-specific embeddings.

---

### 9. Weak AND Retrieval

Efficient lexical retrieval for long queries.

```sql
select * from doc
where weakAnd(default contains "transformer attention neural networks")
```

Often combined with vector search.

**Use case:** Large collections with verbose user queries.

---

## Common Query Parameters

| Parameter | Description |
|-----------|-------------|
| `input.query(q)` | Query embedding tensor |
| `targetHits` | Number of ANN candidates to retrieve |
| `hits` | Number of results returned |
| `ranking` | Rank profile to use |
| `approximate` | Enable (`true`) or disable (`false`) approximate ANN search |
| `timeout` | Maximum query execution time |

---

## Summary of Common Retrieval Patterns

| Retrieval Pattern | Candidate Retrieval | Ranking | Typical Use Case |
|-------------------|--------------------|---------|------------------|
| **BM25** | `userQuery()` | BM25 | Traditional keyword search |
| **Semantic (ANN)** | `nearestNeighbor()` | Closeness | Pure semantic search |
| **Hybrid** | `userQuery() OR nearestNeighbor()` | BM25 + Closeness | General-purpose search engines |
| **BM25 + Re-ranking** | `userQuery()` | BM25 + Closeness | Improve keyword ranking using embeddings |
| **Filtered ANN** | Metadata filter + `nearestNeighbor()` | Closeness | Category- or tenant-specific semantic search |
| **Multi-vector** | Multiple `nearestNeighbor()` clauses | Combined similarity | Multiple embedding fields |
| **Weak AND + ANN** | `weakAnd()` + `nearestNeighbor()` | BM25 + Closeness | Very large document collections |
| **Field-specific Query Embeddings** | Multiple query tensors | Combined similarity | Multi-modal or field-aware retrieval |

### Choosing a Retrieval Strategy

| Goal | Recommended Query |
|------|-------------------|
| Fast keyword search | `userQuery()` |
| Pure semantic search | `nearestNeighbor()` |
| Best overall retrieval quality | `userQuery() OR nearestNeighbor()` |
| Keep BM25 but improve ranking | `userQuery()` + semantic rank profile |
| Search within filtered subsets | `filter AND nearestNeighbor()` |
| Search across multiple embeddings | Multiple `nearestNeighbor()` clauses |